# Trees, forests, and boosting

A shallow tree creates a few if–then regions. A forest averages many randomized trees. Gradient boosting adds small trees in sequence to correct the current residual errors.

## 1. Using standard libraries
Fit a tree, forest, and gradient-boosting model with scikit-learn.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import mean_squared_error

rng = np.random.default_rng(5)
X = rng.uniform(-2, 2, size=(300, 1))
y = np.sin(3 * X[:, 0]) + 0.25 * X[:, 0] + rng.normal(0, 0.18, len(X))
train, test = np.arange(220), np.arange(220, 300)
models = {
    'shallow tree': DecisionTreeRegressor(max_depth=2, random_state=1),
    'forest': RandomForestRegressor(n_estimators=100, max_depth=4, random_state=1),
    'boosting': GradientBoostingRegressor(n_estimators=50, max_depth=2, learning_rate=.08, random_state=1)
}
grid = np.linspace(-2, 2, 400)[:, None]
plt.figure(figsize=(9, 5))
plt.scatter(X[test], y[test], s=20, alpha=.45, label='later observations')
for name, model in models.items():
    model.fit(X[train], y[train])
    mse = mean_squared_error(y[test], model.predict(X[test]))
    print(f'{name:13s} test MSE: {mse:.3f}')
    plt.plot(grid, model.predict(grid), label=name)
plt.title('Different ways to combine simple rules')
plt.xlabel('feature'); plt.ylabel('target'); plt.legend(); plt.show();

## 2. Build boosting from scratch
Fit one-dimensional decision stumps to the residuals, one correction at a time.

In [ ]:
x_train = X[train, 0]
prediction = np.full(len(train), y[train].mean())
stumps = []
for _ in range(50):
    residual = y[train] - prediction
    best = None
    for threshold in np.linspace(-1.8, 1.8, 35):
        left = residual[x_train <= threshold].mean()
        right = residual[x_train > threshold].mean()
        update = np.where(x_train <= threshold, left, right)
        error = np.mean((residual - update) ** 2)
        if best is None or error < best[0]:
            best = (error, threshold, left, right, update)
    _, threshold, left, right, update = best
    prediction += .08 * update
    stumps.append((threshold, left, right))

test_prediction = np.full(len(test), y[train].mean())
for threshold, left, right in stumps:
    test_prediction += .08 * np.where(X[test, 0] <= threshold, left, right)
print(f'From-scratch boosting test MSE: {mean_squared_error(y[test], test_prediction):.3f}')

## What to notice
The shallow tree is visibly piecewise constant. The forest smooths instability by averaging. Boosting builds a flexible curve by adding corrections—but its performance still has to be judged on observations that did not choose the model.